# 1. Setup

In [ ]:
from pandas.core.frame import DataFrame
import pandas as pd
import os
from vnstock import Vnstock

symbol = "POT"
daily_file = f"new\processed_{symbol}.csv"
df: DataFrame = pd.read_csv(daily_file, parse_dates=["time"])

# 2. Download Data from VNSTOCK3

In [ ]:
stock = Vnstock().stock(symbol=symbol, source='TCBS')

df_cash_flow = stock.finance.cash_flow(period='quarter') 
df_income_statement = stock.finance.income_statement(period='quarter')
df_ratio = stock.finance.ratio(period='quarter') 

df_cash_flow.to_csv(f'new/{symbol}_cash_flow.csv', index=False)
df_income_statement.to_csv(f'new/{symbol}_income_statement.csv', index=False)
df_ratio.to_csv(f'new/{symbol}_ratio.csv', index=False)
print('done!')

In [ ]:
len(df)

In [ ]:
df.columns, len(df.columns)

In [ ]:
df = df.pivot(index='time', columns='Symbol')
df.columns = ['{}_{}'.format(sym, feat) for feat, sym in df.columns]
df.reset_index(inplace=True)
df.head()

In [ ]:
len(df)

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
symbol

In [ ]:
df.columns = [
    col.replace(f'{symbol}_', '') if col.startswith(f'{symbol}_') else col
    for col in df.columns
]

In [ ]:
len(df.columns)

In [ ]:
# Map for converting quarter to its corresponding end date string
quarter_end_map = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}

def load_and_process_quarterly(file_path, start_date, end_date):
    df = pd.read_csv(file_path)    
    df["Quarter_End"] = pd.to_datetime(
        df["year"].astype(str) + df["quarter"].astype(int).map(quarter_end_map)
    )
    
    df = df[(df["Quarter_End"] >= start_date) & (df["Quarter_End"] <= end_date)]
    
    df.sort_values("Quarter_End", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

In [ ]:
cash_flow_file = f"new\\{symbol}_cash_flow.csv"
financial_reports_file = f"new\\{symbol}_income_statement.csv"
stock_ratio_file = f"new\\{symbol}_ratio.csv"

start_date = pd.to_datetime("2018-01-01")
end_date   = pd.to_datetime("2025-01-31")

df_cash_flow = load_and_process_quarterly(cash_flow_file, start_date, end_date)
df_financial = load_and_process_quarterly(financial_reports_file, start_date, end_date)
df_stock_ratio = load_and_process_quarterly(stock_ratio_file, start_date, end_date)

In [ ]:
df = df.sort_values("time").reset_index(drop=True)
df_cash_flow = df_cash_flow.sort_values("Quarter_End").reset_index(drop=True)
df_financial = df_financial.sort_values("Quarter_End").reset_index(drop=True)
df_stock_ratio = df_stock_ratio.sort_values("Quarter_End").reset_index(drop=True)

In [ ]:
df_cash_flow.columns

In [ ]:
cols_to_convert = ["invest_cost", "from_invest", "from_financial", "from_sale", "free_cash_flow"]
df_cash_flow[cols_to_convert] = df_cash_flow[cols_to_convert].apply(lambda x: x * 1_000)

In [ ]:
df_financial.columns

In [ ]:
monetary_cols = [
    "revenue",
    # "cost_of_good_sold",
    # "gross_profit",
    "operation_expense",
    "operation_profit",
    # "interest_expense",
    "pre_tax_profit",
    "post_tax_profit",
    "share_holder_income",
    # "ebitda"
]
df_financial[monetary_cols] = df_financial[monetary_cols].apply(lambda x: x * 1_000)

In [ ]:
daily_range = pd.date_range(start=df['time'].min(), end=df['time'].max(), freq='D')
print("Daily range:", daily_range)

def reindex_quarterly_to_daily(df_quarterly, date_col="Quarter_End"):
    df_quarterly = df_quarterly.drop_duplicates(subset=[date_col])
    df_daily = df_quarterly.set_index(date_col).reindex(daily_range, method='ffill')
    df_daily = df_daily.reset_index().rename(columns={'index': 'time'})
    return df_daily

df_cf_daily = reindex_quarterly_to_daily(df_cash_flow, "Quarter_End")
df_financial_daily = reindex_quarterly_to_daily(df_financial, "Quarter_End")
df_stock_ratio_daily = reindex_quarterly_to_daily(df_stock_ratio, "Quarter_End")

df_merged_alt = pd.merge(df, df_cf_daily, on="time", how="left")
df_merged_alt = df_merged_alt.fillna(method='bfill')

df_merged_alt = pd.merge(df_merged_alt, df_financial_daily, on="time", how="left")
df_merged_alt = df_merged_alt.fillna(method='bfill')

df_merged_alt = pd.merge(df_merged_alt, df_stock_ratio_daily, on="time", how="left")
df_merged_alt = df_merged_alt.fillna(method='bfill')

In [ ]:
df_merged_alt.isna().sum()[df_merged_alt.isna().sum() > 0]

In [ ]:
df_merged_alt.head()

In [ ]:
len(df_merged_alt)

In [ ]:
cols_to_drop = ['quarter_x', 'year_x', 'quarter_y', 'year_y', 'quarter', 'year']
df_merged_alt.drop(columns=cols_to_drop, inplace=True)

# 3. Further Processing

In [ ]:
features_vnd = [
    'open', 'VN30_open', 'VNINDEX_open', 'high', 'VN30_high', 'VNINDEX_high', 'low', 'VN30_low', 'VNINDEX_low',
    'close', 'VN30_close', 'VNINDEX_close', 'volume', 'VN30_volume', 'VNINDEX_volume', 'daily_liquidity',
    'VN30_daily_liquidity', 'VNINDEX_daily_liquidity', 'weekly_liquidity', 'VN30_weekly_liquidity', 
    'VNINDEX_weekly_liquidity', 'monthly_liquidity', 'VN30_monthly_liquidity', 'VNINDEX_monthly_liquidity',
    'wma_3', 'VN30_wma_3', 'VNINDEX_wma_3', 'wma_7', 'VN30_wma_7', 'VNINDEX_wma_7', 'wma_14', 'VN30_wma_14', 
    'VNINDEX_wma_14', 'wma_21', 'VN30_wma_21', 'VNINDEX_wma_21', 'wma_50', 'VN30_wma_50', 'VNINDEX_wma_50', 
    'wma_100', 'VN30_wma_100', 'VNINDEX_wma_100', 'obv', 'VN30_obv', 'VNINDEX_obv', 'sma_3', 'VN30_sma_3', 
    'VNINDEX_sma_3', 'sma_7', 'VN30_sma_7', 'VNINDEX_sma_7', 'sma_14', 'VN30_sma_14', 'VNINDEX_sma_14', 
    'sma_21', 'VN30_sma_21', 'VNINDEX_sma_21', 'sma_50', 'VN30_sma_50', 'VNINDEX_sma_50', 'sma_100', 
    'VN30_sma_100', 'VNINDEX_sma_100', 'ema_6', 'VN30_ema_6', 'VNINDEX_ema_6', 'ema_12', 'VN30_ema_12', 
    'VNINDEX_ema_12', 'atr_14', 'VN30_atr_14', 'VNINDEX_atr_14', 'mom_1', 'VN30_mom_1', 'VNINDEX_mom_1', 
    'mom_3', 'VN30_mom_3', 'VNINDEX_mom_3', 'out_macd', 'VN30_out_macd', 'VNINDEX_out_macd', 
    'out_macd_signal', 'VN30_out_macd_signal', 'VNINDEX_out_macd_signal', 'out_macd_hist', 
    'VN30_out_macd_hist', 'VNINDEX_out_macd_hist', 'tsf_10', 'VN30_tsf_10', 'VNINDEX_tsf_10', 
    'tsf_20', 'VN30_tsf_20', 'VNINDEX_tsf_20', 'bbandsmiddle', 'VN30_bbandsmiddle', 'VNINDEX_bbandsmiddle', 
    'bbandsupper', 'VN30_bbandsupper', 'VNINDEX_bbandsupper', 'bbandslower', 'VN30_bbandslower', 
    'VNINDEX_bbandslower', 'invest_cost', 'from_invest', 'from_financial', 'from_sale', 'free_cash_flow', 
    'revenue', 'operation_expense', 'operation_profit', 'pre_tax_profit', 'post_tax_profit', 
    'share_holder_income', 'earning_per_share', 'book_value_per_share'
]
# Fix the incorrect column names by replacing "liquid#ity" with "liquidity"
df_merged_alt.columns = df_merged_alt.columns.str.replace("weekly_liquid#ity", "weekly_liquidity")
df_merged_alt.columns = df_merged_alt.columns.str.replace("VN30_weekly_liquid#ity", "VN30_weekly_liquidity")
df_merged_alt.columns = df_merged_alt.columns.str.replace("VNINDEX_weekly_liquid#ity", "VNINDEX_weekly_liquidity")

df_merged_alt.rename(columns={col: f"{col}_kVND" for col in features_vnd if col in df_merged_alt.columns}, inplace=True)

In [ ]:
len(df_merged_alt.columns.tolist())

In [ ]:
sorted(df_merged_alt.columns.tolist())

In [ ]:
all_features = ['time', 'daily_return', 'VN30_daily_return', 'VNINDEX_daily_return', 'weekly_return', 'VN30_weekly_return', 'VNINDEX_weekly_return', 'monthly_return', 'VN30_monthly_return', 'VNINDEX_monthly_return', 'daily_volatility', 'VN30_daily_volatility', 'VNINDEX_daily_volatility', 'weekly_volatility', 'VN30_weekly_volatility', 'VNINDEX_weekly_volatility', 'monthly_volatility', 'VN30_monthly_volatility', 'VNINDEX_monthly_volatility', 'high_minus_close', 'VN30_high_minus_close', 'VNINDEX_high_minus_close', 'low_minus_open', 'VN30_low_minus_open', 'VNINDEX_low_minus_open', 'cumulative_return', 'VN30_cumulative_return', 'VNINDEX_cumulative_return', 'rsi_6', 'VN30_rsi_6', 'VNINDEX_rsi_6', 'rsi_12', 'VN30_rsi_12', 'VNINDEX_rsi_12', 'rsi_14', 'VN30_rsi_14', 'VNINDEX_rsi_14', 'stoch_rsi_6', 'VN30_stoch_rsi_6', 'VNINDEX_stoch_rsi_6', 'stoch_rsi_12', 'VN30_stoch_rsi_12', 'VNINDEX_stoch_rsi_12', 'stoch_rsi_14', 'VN30_stoch_rsi_14', 'VNINDEX_stoch_rsi_14', 'mfi_14', 'VN30_mfi_14', 'VNINDEX_mfi_14', 'adx_14', 'VN30_adx_14', 'VNINDEX_adx_14', 'adx_20', 'VN30_adx_20', 'VNINDEX_adx_20', 'cci_12', 'VN30_cci_12', 'VNINDEX_cci_12', 'cci_20', 'VN30_cci_20', 'VNINDEX_cci_20', 'rocr_3', 'VN30_rocr_3', 'VNINDEX_rocr_3', 'rocr_12', 'VN30_rocr_12', 'VNINDEX_rocr_12', 'willr', 'VN30_willr', 'VNINDEX_willr', 'trix', 'VN30_trix', 'VNINDEX_trix', 'year_revenue_growth', 'quarter_revenue_growth', 'year_operation_profit_growth', 'quarter_operation_profit_growth', 'year_share_holder_income_growth', 'quarter_share_holder_income_growth', 'price_to_earning', 'price_to_book', 'roe', 'roa', 'equity_on_total_asset', 'equity_on_liability', 'eps_change', 'asset_on_equity', 'payable_on_equity', 'book_value_per_share_change', 'open_kVND', 'VN30_open_kVND', 'VNINDEX_open_kVND', 'high_kVND', 'VN30_high_kVND', 'VNINDEX_high_kVND', 'low_kVND', 'VN30_low_kVND', 'VNINDEX_low_kVND', 'close_kVND', 'VN30_close_kVND', 'VNINDEX_close_kVND', 'volume_kVND', 'VN30_volume_kVND', 'VNINDEX_volume_kVND', 'daily_liquidity_kVND', 'VN30_daily_liquidity_kVND', 'VNINDEX_daily_liquidity_kVND', 'weekly_liquidity_kVND', 'VN30_weekly_liquidity_kVND', 'VNINDEX_weekly_liquidity_kVND', 'monthly_liquidity_kVND', 'VN30_monthly_liquidity_kVND', 'VNINDEX_monthly_liquidity_kVND', 'wma_3_kVND', 'VN30_wma_3_kVND', 'VNINDEX_wma_3_kVND', 'wma_7_kVND', 'VN30_wma_7_kVND', 'VNINDEX_wma_7_kVND', 'wma_14_kVND', 'VN30_wma_14_kVND', 'VNINDEX_wma_14_kVND', 'wma_21_kVND', 'VN30_wma_21_kVND', 'VNINDEX_wma_21_kVND', 'wma_50_kVND', 'VN30_wma_50_kVND', 'VNINDEX_wma_50_kVND', 'wma_100_kVND', 'VN30_wma_100_kVND', 'VNINDEX_wma_100_kVND', 'obv_kVND', 'VN30_obv_kVND', 'VNINDEX_obv_kVND', 'sma_3_kVND', 'VN30_sma_3_kVND', 'VNINDEX_sma_3_kVND', 'sma_7_kVND', 'VN30_sma_7_kVND', 'VNINDEX_sma_7_kVND', 'sma_14_kVND', 'VN30_sma_14_kVND', 'VNINDEX_sma_14_kVND', 'sma_21_kVND', 'VN30_sma_21_kVND', 'VNINDEX_sma_21_kVND', 'sma_50_kVND', 'VN30_sma_50_kVND', 'VNINDEX_sma_50_kVND', 'sma_100_kVND', 'VN30_sma_100_kVND', 'VNINDEX_sma_100_kVND', 'ema_6_kVND', 'VN30_ema_6_kVND', 'VNINDEX_ema_6_kVND', 'ema_12_kVND', 'VN30_ema_12_kVND', 'VNINDEX_ema_12_kVND', 'atr_14_kVND', 'VN30_atr_14_kVND', 'VNINDEX_atr_14_kVND', 'mom_1_kVND', 'VN30_mom_1_kVND', 'VNINDEX_mom_1_kVND', 'mom_3_kVND', 'VN30_mom_3_kVND', 'VNINDEX_mom_3_kVND', 'out_macd_kVND', 'VN30_out_macd_kVND', 'VNINDEX_out_macd_kVND', 'out_macd_signal_kVND', 'VN30_out_macd_signal_kVND', 'VNINDEX_out_macd_signal_kVND', 'out_macd_hist_kVND', 'VN30_out_macd_hist_kVND', 'VNINDEX_out_macd_hist_kVND', 'tsf_10_kVND', 'VN30_tsf_10_kVND', 'VNINDEX_tsf_10_kVND', 'tsf_20_kVND', 'VN30_tsf_20_kVND', 'VNINDEX_tsf_20_kVND', 'bbandsmiddle_kVND', 'VN30_bbandsmiddle_kVND', 'VNINDEX_bbandsmiddle_kVND', 'bbandsupper_kVND', 'VN30_bbandsupper_kVND', 'VNINDEX_bbandsupper_kVND', 'bbandslower_kVND', 'VN30_bbandslower_kVND', 'VNINDEX_bbandslower_kVND', 'invest_cost_kVND', 'from_invest_kVND', 'from_financial_kVND', 'from_sale_kVND', 'free_cash_flow_kVND', 'revenue_kVND', 'operation_expense_kVND', 'operation_profit_kVND', 'pre_tax_profit_kVND', 'post_tax_profit_kVND', 'share_holder_income_kVND', 'earning_per_share_kVND', 'book_value_per_share_kVND']

df_merged_alt = df_merged_alt[all_features]

In [ ]:
len(df_merged_alt.columns.tolist())

In [ ]:
df_merged_alt.isna().sum()[df_merged_alt.isna().sum() > 0]

In [ ]:
output_file = f"finals/feature_engineered_{symbol}.csv"
df_merged_alt.to_csv(output_file, index=False)
print(f"Saved combined feature-engineered file to {output_file}")

In [ ]:
files_to_delete = [
    f"new/{symbol}_cash_flow.csv",
    f"new/{symbol}_income_statement.csv",
    f"new/{symbol}_ratio.csv",
    f"new/processed_{symbol}.csv",
    f"new/{symbol}.csv",
]

for file in files_to_delete:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted file: {file}")

# The End